In [2]:
import pandas as pd

inventory_gold = pd.read_parquet(
    "data/gold/inventory_gold_2026-02-01.parquet"
)

In [3]:
inventory_gold.shape, inventory_gold[["inventory_units","inventory_value","sales_12m","days_inventory"]].describe()

((504, 12),
        inventory_units  inventory_value  sales_12m  days_inventory
 count       504.000000       293.000000      504.0           219.0
 mean        666.615079     65617.180994        0.0             0.0
 std         597.422152     99162.505237        0.0             0.0
 min           0.000000         0.000000        0.0             0.0
 25%         144.000000         0.000000        0.0             0.0
 50%         607.000000     13976.976000        0.0             0.0
 75%        1029.000000     90904.625600        0.0             0.0
 max        1911.000000    616360.194000        0.0             0.0)

In [4]:
inventory_gold[[
    "sku_id",
    "inventory_units",
    "inventory_value",
    "sales_12m",
    "days_inventory"
]].head()

,sku_id,inventory_units,inventory_value,sales_12m,days_inventory
0,1,1085.0,NaN,0,NaN
1,2,1109.0,NaN,0,NaN
2,3,1352.0,NaN,0,NaN
3,4,1322.0,NaN,0,NaN
4,316,1361.0,NaN,0,NaN


In [5]:
inventory_gold["inventory_turnover"] = (
    inventory_gold["sales_12m"] / inventory_gold["inventory_value"]
)

In [6]:
TARGET_DAYS = 90      # objetivo sano
MAX_DAYS = 180        # alarma roja

In [7]:
def inventory_flag(days):
    if days <= TARGET_DAYS:
        return "🟢"
    elif days <= MAX_DAYS:
        return "🟡"
    else:
        return "🔴"

inventory_gold["inventory_flag"] = inventory_gold["days_inventory"].apply(inventory_flag)

In [8]:
inventory_gold["excess_days"] = (
    inventory_gold["days_inventory"] - TARGET_DAYS
).clip(lower=0)

inventory_gold["capital_trapped"] = (
    inventory_gold["inventory_value"] *
    inventory_gold["excess_days"] /
    inventory_gold["days_inventory"]
)

In [9]:
inventory_gold.sort_values(
    "capital_trapped",
    ascending=False
)[[
    "sku_id",
    "inventory_flag",
    "days_inventory",
    "inventory_value",
    "capital_trapped"
]].head(15)

,sku_id,inventory_flag,days_inventory,inventory_value,capital_trapped
0,1,🔴,NaN,NaN,NaN
1,2,🔴,NaN,NaN,NaN
2,3,🔴,NaN,NaN,NaN
3,4,🔴,NaN,NaN,NaN
4,316,🔴,NaN,NaN,NaN
5,317,🔴,NaN,NaN,NaN
6,318,🔴,NaN,NaN,NaN
7,319,🔴,NaN,NaN,NaN
8,320,🔴,NaN,NaN,NaN
9,321,🔴,NaN,NaN,NaN
